<a href="https://www.kaggle.com/code/xpertdl/poverty-prediction-challenge?scriptVersionId=296485207" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:

from sklearn.preprocessing import LabelEncoder, StandardScaler

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pickle
import zipfile


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df_train_X = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/train_hh_features.csv') # The Questions (X)
df_train_y = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/train_hh_gt.csv')       # The Answers (y)
df_test_X = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/test_hh_features.csv')       # The Answers (y)
df_desc = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/feature_descriptions.csv')       # The Answers (y)

In [ ]:
df_train_X.shape

In [ ]:
df_train_X.columns

In [ ]:
df_train_y.head()

In [ ]:
train_df = pd.merge(df_train_X, df_train_y, on='hhid')

In [ ]:
train_df.head()

In [ ]:
# 1. Load Data
train_features = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/train_hh_features.csv')
train_labels = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/train_hh_gt.csv')
test_features = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/test_hh_features.csv')

# 2. Merge Train with Target
# We join on 'household_id' (which is 'hhid' in your description, but let's stick to the CSV header)
# NOTE: Check if your CSV header says 'hhid' or 'household_id'. 
# Based on your sample, it says 'hhid'.
train_df = pd.merge(train_features, train_labels, on='hhid')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_features.shape}")

# 3. Combine for consistent encoding
train_df['is_train'] = 1
test_features['is_train'] = 0
test_features['cons_ppp17'] = 0 # Dummy
test_features['weight'] = 0     # Dummy

all_data = pd.concat([train_df, test_features], axis=0, ignore_index=True)

# --- CLEANING RULES ---

# Rule 1: Drop useless ID columns
drop_cols = ['hhid', 'com', 'strata', 'survey_id', 'survey_id_x', 'survey_id_y', 'sample']
# We only drop if they exist in the dataframe
all_data = all_data.drop([c for c in drop_cols if c in all_data.columns], axis=1)

# Rule 2: Convert "Yes/No" Columns to 1/0
# This includes 'consumed' columns and 'any_nonagric', 'urban' etc if they are Yes/No
yes_no_cols = [c for c in all_data.columns if 'consumed' in c]
print(f"Converting {len(yes_no_cols)} consumed columns...")
for col in yes_no_cols:
    all_data[col] = (all_data[col] == 'Yes').astype(int)

# Rule 3: Define Categorical vs Numerical
# We manually list the text columns based on your sample
text_cols = [
    'male', 'owner', 'water', 'toilet', 'sewer', 'elect', 
    'water_source', 'sanitation_source', 'dweltyp', 'employed', 
    'educ_max', 'any_nonagric', 'sector1d', 'urban'
]

# All other columns (except target/weight/is_train) are Numerical
ignore = ['cons_ppp17', 'weight', 'is_train'] + yes_no_cols + text_cols
num_cols = [c for c in all_data.columns if c not in ignore]

print(f"Text Categories: {len(text_cols)}")
print(f"Numerical Features: {len(num_cols)}")

# Rule 4: Fill Missing Values
all_data[text_cols] = all_data[text_cols].fillna("MISSING")
all_data[num_cols] = all_data[num_cols].fillna(0)

# Rule 5: Encode Text Categories
embedding_sizes = []
for col in text_cols:
    all_data[col] = all_data[col].astype(str) # Ensure string
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col])
    
    # Calculate Embedding Size
    n_unique = len(le.classes_)
    emb_dim = min(50, (n_unique + 1) // 2)
    embedding_sizes.append((n_unique, emb_dim))

# Rule 6: Scale Numerical Values (Crucial for utl_exp_ppp17)
scaler = StandardScaler()
all_data[num_cols] = scaler.fit_transform(all_data[num_cols])

# Rule 7: Split Back
train_final = all_data[all_data['is_train'] == 1].copy()
test_final = all_data[all_data['is_train'] == 0].copy()

# Save Everything
train_final.to_csv('/kaggle/working/train_processed.csv', index=False)
test_final.to_csv('/kaggle/working/test_processed.csv', index=False)

with open('/kaggle/working//metadata.pkl', 'wb') as f:
    pickle.dump({
        'text_cols': text_cols,
        'num_cols': num_cols, # Includes the yes/no columns now? No, we need to add them.
        'yes_no_cols': yes_no_cols,
        'emb_sizes': embedding_sizes
    }, f)

print("✅ Data Processed Successfully!")

In [ ]:
df_train_X = pd.read_csv('/kaggle/working/train_processed.csv') # The Questions (X)
# df_train_y = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/train_hh_gt.csv')       # The Answers (y)
# df_test_X = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/test_hh_features.csv')       # The Answers (y)
# df_desc = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/feature_descriptions.csv') 

In [ ]:
df_train_X.head()

In [ ]:
df_test_X = pd.read_csv('/kaggle/working/test_processed.csv')

In [ ]:
df_test_X.head()

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 512
EPOCHS = 50
LR = 0.01

DEVICE

In [ ]:
train_df = pd.read_csv('/kaggle/working/train_processed.csv')
with open('/kaggle/working/metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

text_cols = meta['text_cols']
# Combine "Numerical" + "Yes/No" into one big numerical input vector
input_num_cols = meta['num_cols'] + meta['yes_no_cols']
emb_sizes = meta['emb_sizes']

In [ ]:
class PovertyDataset(Dataset):
    def __init__(self, df, text_cols, num_cols):
        self.text = torch.tensor(df[text_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.target = torch.tensor(df['cons_ppp17'].values, dtype=torch.float32)
        self.weight = torch.tensor(df['weight'].values, dtype=torch.float32)

    def __len__(self): return len(self.target)
    def __getitem__(self, idx):
        return self.text[idx], self.num[idx], self.target[idx], self.weight[idx]

train_ds = PovertyDataset(train_df, text_cols, input_num_cols)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class PovertyModel(nn.Module):
    def __init__(self, emb_sizes, n_num):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, d) for n, d in emb_sizes])
        n_emb_out = sum(d for n, d in emb_sizes)
        
        self.net = nn.Sequential(
            nn.Linear(n_emb_out + n_num, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1) # Output: Consumption
        )
        
    def forward(self, x_text, x_num):
        # Process Embeddings
        embs = [e(x_text[:, i]) for i, e in enumerate(self.embs)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.net(x)

In [ ]:
model = PovertyModel(emb_sizes, len(input_num_cols)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.L1Loss(reduction='none')

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for text, num, target, weight in train_loader:
        text, num, target, weight = text.to(DEVICE), num.to(DEVICE), target.to(DEVICE), weight.to(DEVICE)
        
        optimizer.zero_grad()
        pred = model(text, num).squeeze()
        
        # Weighted MAE Loss
        loss = (criterion(pred, target) * weight).mean()
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
torch.save(model.state_dict(), "poverty_model.pth")
print("✅ Model Trained and Saved!")

In [ ]:
BATCH_SIZE = 1024

In [ ]:
test_df = pd.read_csv('/kaggle/working/test_processed.csv')
with open('/kaggle/working/metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

# Reconstruct Input Columns just like Training
text_cols = meta['text_cols']
input_num_cols = meta['num_cols'] + meta['yes_no_cols'] # Combine Number + Yes/No
emb_sizes = meta['emb_sizes']

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, text_cols, num_cols):
        self.text = torch.tensor(df[text_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.ids = df['household_id'].values # Keep IDs for submission

    def __len__(self): return len(self.text)
    def __getitem__(self, idx):
        return self.text[idx], self.num[idx], self.ids[idx]

test_ds = TestDataset(test_df, text_cols, input_num_cols)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# 3. Load Model Architecture (Must match Train)
class PovertyModel(nn.Module):
    def __init__(self, emb_sizes, n_num):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, d) for n, d in emb_sizes])
        n_emb_out = sum(d for n, d in emb_sizes)
        self.net = nn.Sequential(
            nn.Linear(n_emb_out + n_num, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x_text, x_num):
        embs = [e(x_text[:, i]) for i, e in enumerate(self.embs)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.net(x)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 1024 

print(f"🚀 Generating Submission on {DEVICE}...")

# 1. Load PROCESSED Data (For the Model)
print("Loading Processed Features...")
test_df = pd.read_csv('/kaggle/working/test_processed.csv')

# 2. Load RAW Data (To get 'hhid' and 'weight' which we dropped earlier)
print("Loading Raw IDs...")
raw_test = pd.read_csv('/kaggle/input/twb-poverty-prediction/TWB/feature_descriptions.csv')

# CRITICAL FIX: Ensure we have the IDs and Weights
# The raw file has 'hhid', but the submission needs 'household_id'
ids = raw_test['hhid'].values
weights = raw_test['weight'].values
survey_ids = raw_test['survey_id'].values

# 3. Load Metadata
with open('/kaggle/working/metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

text_cols = meta['text_cols']
# Combine "Numerical" + "Yes/No"
input_num_cols = meta['num_cols'] + meta['yes_no_cols']
emb_sizes = meta['emb_sizes']

# 4. Define Dataset (Features from processed, IDs passed separately)
class TestDataset(Dataset):
    def __init__(self, df, text_cols, num_cols):
        self.text = torch.tensor(df[text_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)

    def __len__(self): return len(self.text)
    def __getitem__(self, idx):
        return self.text[idx], self.num[idx]

test_ds = TestDataset(test_df, text_cols, input_num_cols)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# 5. Load Model Architecture
class PovertyModel(nn.Module):
    def __init__(self, emb_sizes, n_num):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, d) for n, d in emb_sizes])
        n_emb_out = sum(d for n, d in emb_sizes)
        self.net = nn.Sequential(
            nn.Linear(n_emb_out + n_num, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x_text, x_num):
        embs = [e(x_text[:, i]) for i, e in enumerate(self.embs)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.net(x)

model = PovertyModel(emb_sizes, len(input_num_cols)).to(DEVICE)
model.load_state_dict(torch.load("/kaggle/working/poverty_model.pth"))
model.eval()

# 6. Predict
print("Predicting...")
all_preds = []

with torch.no_grad():
    for text, num in test_loader:
        text, num = text.to(DEVICE), num.to(DEVICE)
        pred = model(text, num).squeeze()
        pred = torch.relu(pred) # No negative money
        all_preds.extend(pred.cpu().numpy())

# 7. Create Results DataFrame
results = pd.DataFrame({
    'household_id': ids,      # From Raw File (hhid)
    'survey_id': survey_ids,  # From Raw File
    'weight': weights,        # From Raw File
    'pred_consumption': all_preds
})

# --- FILE 1: Consumption ---
print("Creating File 1: Consumption...")
sub_cons = results[['survey_id', 'household_id', 'pred_consumption']].copy()
# Rename for competition format
sub_cons.columns = ['survey_id', 'household_id', 'per_capita_household_consumption']
sub_cons.to_csv('predicted_household_consumption.csv', index=False)

# --- FILE 2: Poverty Rates ---
print("Creating File 2: Poverty Rates...")
THRESHOLDS = [
    3.17, 3.94, 4.60, 5.26, 5.88, 6.47, 7.06, 7.70, 8.40, 
    9.13, 9.87, 10.70, 11.62, 12.69, 14.03, 15.64, 17.76, 20.99, 27.37
]

output_rows = []
for survey in results['survey_id'].unique():
    survey_data = results[results['survey_id'] == survey]
    total_pop = survey_data['weight'].sum()
    
    survey_rates = [survey]
    for thresh in THRESHOLDS:
        poor_pop = survey_data[survey_data['pred_consumption'] < thresh]['weight'].sum()
        survey_rates.append(poor_pop / total_pop)
        
    output_rows.append(survey_rates)

cols = ['survey_id'] + [f'pct_hh_below_{t}' for t in THRESHOLDS]
sub_poverty = pd.DataFrame(output_rows, columns=cols)
sub_poverty.to_csv('predicted_poverty_distribution.csv', index=False)

# Zip
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('predicted_household_consumption.csv')
    z.write('predicted_poverty_distribution.csv')

print("\n✅ DONE! 'submission.zip' created.")

In [ ]:
"""
================================================================================
SOLUTION V3 - NEURAL NETWORK + LIGHTGBM ENSEMBLE
================================================================================

WHY ENSEMBLE?
- Neural Networks: Good at capturing complex non-linear patterns
- LightGBM: Excellent for tabular data, handles features interactions naturally
- Combining both = More robust predictions

This is a WINNING STRATEGY used in most Kaggle/DrivenData competitions!

================================================================================
"""

import os
import warnings
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ================================================================================
# CONFIG
# ================================================================================
class Config:
    DATA_DIR = Path("/kaggle/input/twb-poverty-prediction/TWB")
    OUTPUT_DIR = Path("/kaggle/working/")

    THRESHOLDS = [3.17, 3.94, 4.60, 5.26, 5.88, 6.47, 7.06, 7.70, 8.40,
                  9.13, 9.87, 10.70, 11.62, 12.69, 14.03, 15.64, 17.76, 20.99, 27.37]

    # Neural Network params
    NN_BATCH_SIZE = 512
    NN_LR = 0.001
    NN_EPOCHS = 60
    NN_EARLY_STOP = 10

    # LightGBM params - optimized for this task
    LGB_PARAMS = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 63,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 20,
        'verbose': -1,
        'seed': SEED,
        'n_jobs': -1
    }

    N_FOLDS = 5

    # Ensemble weights (can be tuned)
    NN_WEIGHT = 0.4  # 40% Neural Network
    LGB_WEIGHT = 0.6  # 60% LightGBM (usually better for tabular)

    USE_LOG_TARGET = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {Config.DEVICE}")
print(f"Ensemble: {Config.NN_WEIGHT*100:.0f}% NN + {Config.LGB_WEIGHT*100:.0f}% LightGBM")

# ================================================================================
# DATA LOADING
# ================================================================================
def load_data():
    print("\n" + "="*60)
    print("STEP 1: LOADING DATA")
    print("="*60)

    train_features = pd.read_csv(Config.DATA_DIR / "train_hh_features.csv")
    train_gt = pd.read_csv(Config.DATA_DIR / "train_hh_gt.csv")
    train_rates = pd.read_csv(Config.DATA_DIR / "train_rates_gt.csv")
    test_features = pd.read_csv(Config.DATA_DIR / "test_hh_features.csv")

    train_data = train_features.merge(train_gt, on=['hhid'], suffixes=('', '_gt'))
    if 'survey_id_gt' in train_data.columns:
        train_data['survey_id'] = train_data['survey_id_gt']
        train_data.drop(columns=['survey_id_gt'], inplace=True)

    print(f"Train: {train_data.shape}, Test: {test_features.shape}")
    return train_data, test_features, train_rates

# ================================================================================
# FEATURE ENGINEERING
# ================================================================================
def convert_val(val):
    if pd.isna(val): return 0
    s = str(val).strip().lower()
    return 1 if s in ['yes', '1', '1.0', 'true', 'access'] else 0

def engineer_features(train_data, test_data):
    print("\n" + "="*60)
    print("STEP 2: FEATURE ENGINEERING")
    print("="*60)

    train_data['is_train'] = 1
    test_data['is_train'] = 0

    for col in train_data.columns:
        if col not in test_data.columns and col not in ['cons_ppp17', 'is_train']:
            test_data[col] = np.nan

    combined = pd.concat([train_data, test_data], ignore_index=True)

    # Binary columns
    consumed_cols = [c for c in combined.columns if c.startswith('consumed')]
    binary_cols = ['water', 'toilet', 'sewer', 'elect', 'owner', 'male', 'employed', 'any_nonagric']

    for col in consumed_cols + binary_cols:
        if col in combined.columns:
            combined[col] = combined[col].apply(convert_val)

    # Numeric columns
    num_cols = ['num_children5', 'num_children10', 'num_children18', 'num_adult_female',
                'num_adult_male', 'num_elderly', 'hsize', 'utl_exp_ppp17', 'sworkershh',
                'sfworkershh', 'share_secondary', 'age', 'weight', 'strata']
    for col in num_cols:
        if col in combined.columns:
            combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(0)

    # === FEATURE CREATION ===

    # Food features
    combined['total_food'] = combined[consumed_cols].sum(axis=1)
    combined['food_diversity'] = combined['total_food'] / max(len(consumed_cols), 1)

    # Food categories
    protein = ['consumed800', 'consumed900', 'consumed700', 'consumed2000', 'consumed2100']
    carbs = ['consumed100', 'consumed300', 'consumed500', 'consumed1900']
    dairy = ['consumed400', 'consumed2400', 'consumed2700']

    combined['protein_score'] = combined[[c for c in protein if c in combined.columns]].sum(axis=1)
    combined['carbs_score'] = combined[[c for c in carbs if c in combined.columns]].sum(axis=1)
    combined['dairy_score'] = combined[[c for c in dairy if c in combined.columns]].sum(axis=1)

    # Household composition
    combined['total_children'] = combined['num_children5'] + combined['num_children10'] + combined['num_children18']
    combined['total_adults'] = combined['num_adult_female'] + combined['num_adult_male']
    combined['dependency_ratio'] = combined['total_children'] / (combined['total_adults'] + 1)
    combined['elderly_ratio'] = combined['num_elderly'] / (combined['hsize'] + 1)
    combined['child_ratio'] = combined['total_children'] / (combined['hsize'] + 1)

    # Infrastructure
    combined['infra_score'] = combined['water'] + combined['toilet'] + combined['sewer'] + combined['elect']

    # Per capita
    combined['utl_per_capita'] = combined['utl_exp_ppp17'] / (combined['hsize'] + 1)
    combined['utl_per_adult'] = combined['utl_exp_ppp17'] / (combined['total_adults'] + 1)

    # Employment
    combined['employment_score'] = combined['sworkershh'] + combined['sfworkershh']

    # Log transforms
    combined['log_utl'] = np.log1p(combined['utl_exp_ppp17'])
    combined['log_weight'] = np.log1p(combined['weight'])
    combined['log_hsize'] = np.log1p(combined['hsize'])

    # Interaction features (important for LightGBM)
    combined['age_x_employed'] = combined['age'] * combined['employed']
    combined['hsize_x_infra'] = combined['hsize'] * combined['infra_score']
    combined['food_x_infra'] = combined['total_food'] * combined['infra_score']
    combined['adults_x_work'] = combined['total_adults'] * combined['sworkershh']

    # Encode categoricals
    cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
    for col in cat_cols:
        if col != 'is_train':
            combined[col] = combined[col].fillna('Unknown')
            combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

    # Fill NaN
    for col in combined.select_dtypes(include=['int64', 'float64']).columns:
        combined[col] = combined[col].fillna(combined[col].median())

    train_proc = combined[combined['is_train'] == 1].drop(columns=['is_train'])
    test_proc = combined[combined['is_train'] == 0].drop(columns=['is_train', 'cons_ppp17'], errors='ignore')

    print(f"Features created: {train_proc.shape[1] - 3}")
    return train_proc, test_proc

# ================================================================================
# NEURAL NETWORK MODEL
# ================================================================================
class PovertyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# ================================================================================
# DATA PREPARATION
# ================================================================================
def prepare_data(train_data, test_data):
    print("\n" + "="*60)
    print("STEP 3: PREPARING DATA")
    print("="*60)

    exclude = ['hhid', 'survey_id', 'cons_ppp17', 'com']
    features = [c for c in train_data.columns if c not in exclude]

    X_train = train_data[features].values.astype(np.float32)
    y_train = train_data['cons_ppp17'].values.astype(np.float32)
    X_test = test_data[features].values.astype(np.float32)

    # Log transform
    y_train_log = np.log1p(y_train) if Config.USE_LOG_TARGET else y_train

    train_meta = train_data[['survey_id', 'hhid', 'weight']].copy()
    test_meta = test_data[['survey_id', 'hhid', 'weight']].copy()

    # Scale for NN (LightGBM doesn't need scaling)
    scaler = StandardScaler()
    X_train_scaled = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_scaled = np.nan_to_num(scaler.transform(X_test), 0)

    print(f"Features: {len(features)}, Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

    return (X_train, X_train_scaled, y_train, y_train_log,
            X_test, X_test_scaled, train_meta, test_meta, features)

# ================================================================================
# TRAINING FUNCTIONS
# ================================================================================
def train_nn_fold(X_tr, y_tr, X_val, y_val, input_dim, config):
    """Train Neural Network for one fold."""
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
        batch_size=config.NN_BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
        batch_size=config.NN_BATCH_SIZE
    )

    model = PovertyNet(input_dim).to(config.DEVICE)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=config.NN_LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_loss = float('inf')
    patience = 0
    best_state = None

    for epoch in range(config.NN_EPOCHS):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(config.DEVICE), by.to(config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(config.DEVICE), by.to(config.DEVICE)
                val_loss += criterion(model(bx), by).item() * len(bx)
        val_loss /= len(y_val)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict().copy()
            patience = 0
        else:
            patience += 1

        if patience >= config.NN_EARLY_STOP:
            break

    model.load_state_dict(best_state)
    return model

def train_lgb_fold(X_tr, y_tr, X_val, y_val, features, config):
    """Train LightGBM for one fold."""
    train_set = lgb.Dataset(X_tr, y_tr, feature_name=features)
    val_set = lgb.Dataset(X_val, y_val, feature_name=features, reference=train_set)

    model = lgb.train(
        config.LGB_PARAMS,
        train_set,
        num_boost_round=1000,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    return model

def train_ensemble(X_train, X_train_scaled, y_train_log, y_train_orig, features, config):
    """Train both NN and LightGBM with K-Fold CV."""
    print("\n" + "="*60)
    print("STEP 4: TRAINING ENSEMBLE")
    print("="*60)

    kf = KFold(n_splits=config.N_FOLDS, shuffle=True, random_state=SEED)

    nn_models = []
    lgb_models = []
    nn_scores = []
    lgb_scores = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\n--- Fold {fold+1}/{config.N_FOLDS} ---")

        # Split data
        X_tr_raw, X_val_raw = X_train[tr_idx], X_train[val_idx]
        X_tr_scaled, X_val_scaled = X_train_scaled[tr_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_log[tr_idx], y_train_log[val_idx]
        y_val_orig = y_train_orig[val_idx]

        # Train Neural Network
        print("  Training NN...", end=" ")
        nn_model = train_nn_fold(X_tr_scaled, y_tr, X_val_scaled, y_val,
                                  X_train_scaled.shape[1], config)
        nn_models.append(nn_model)

        # NN validation
        nn_model.eval()
        with torch.no_grad():
            nn_preds = nn_model(torch.FloatTensor(X_val_scaled).to(config.DEVICE)).cpu().numpy()
        nn_preds = np.expm1(nn_preds) if config.USE_LOG_TARGET else nn_preds
        nn_mae = mean_absolute_error(y_val_orig, np.maximum(nn_preds, 0.01))
        nn_scores.append(nn_mae)
        print(f"MAE: {nn_mae:.4f}")

        # Train LightGBM
        print("  Training LightGBM...", end=" ")
        lgb_model = train_lgb_fold(X_tr_raw, y_tr, X_val_raw, y_val, features, config)
        lgb_models.append(lgb_model)

        # LightGBM validation
        lgb_preds = lgb_model.predict(X_val_raw)
        lgb_preds = np.expm1(lgb_preds) if config.USE_LOG_TARGET else lgb_preds
        lgb_mae = mean_absolute_error(y_val_orig, np.maximum(lgb_preds, 0.01))
        lgb_scores.append(lgb_mae)
        print(f"MAE: {lgb_mae:.4f}")

        # Ensemble validation
        ensemble_preds = config.NN_WEIGHT * nn_preds + config.LGB_WEIGHT * lgb_preds
        ensemble_mae = mean_absolute_error(y_val_orig, np.maximum(ensemble_preds, 0.01))
        print(f"  Ensemble MAE: {ensemble_mae:.4f}")

    print(f"\n{'='*40}")
    print(f"NN Mean MAE:       {np.mean(nn_scores):.4f} (+/- {np.std(nn_scores):.4f})")
    print(f"LightGBM Mean MAE: {np.mean(lgb_scores):.4f} (+/- {np.std(lgb_scores):.4f})")
    print(f"{'='*40}")

    return nn_models, lgb_models

# ================================================================================
# PREDICTION
# ================================================================================
def predict_ensemble(nn_models, lgb_models, X_test, X_test_scaled, config):
    """Generate ensemble predictions."""
    print("\n" + "="*60)
    print("STEP 5: GENERATING PREDICTIONS")
    print("="*60)

    # NN predictions
    nn_preds_all = []
    for model in nn_models:
        model.eval()
        with torch.no_grad():
            preds = model(torch.FloatTensor(X_test_scaled).to(config.DEVICE)).cpu().numpy()
        nn_preds_all.append(preds)
    nn_preds = np.mean(nn_preds_all, axis=0)
    nn_preds = np.expm1(nn_preds) if config.USE_LOG_TARGET else nn_preds

    # LightGBM predictions
    lgb_preds_all = []
    for model in lgb_models:
        preds = model.predict(X_test)
        lgb_preds_all.append(preds)
    lgb_preds = np.mean(lgb_preds_all, axis=0)
    lgb_preds = np.expm1(lgb_preds) if config.USE_LOG_TARGET else lgb_preds

    # Ensemble
    ensemble_preds = config.NN_WEIGHT * nn_preds + config.LGB_WEIGHT * lgb_preds
    ensemble_preds = np.maximum(ensemble_preds, 0.01)

    print(f"NN predictions:       min={nn_preds.min():.2f}, max={nn_preds.max():.2f}, mean={nn_preds.mean():.2f}")
    print(f"LightGBM predictions: min={lgb_preds.min():.2f}, max={lgb_preds.max():.2f}, mean={lgb_preds.mean():.2f}")
    print(f"Ensemble predictions: min={ensemble_preds.min():.2f}, max={ensemble_preds.max():.2f}, mean={ensemble_preds.mean():.2f}")

    return ensemble_preds

# ================================================================================
# POVERTY RATES & SUBMISSION
# ================================================================================
def calc_poverty_rates(predictions, meta_df, thresholds):
    """Calculate weighted poverty rates."""
    meta_df = meta_df.copy()
    meta_df['pred'] = predictions

    results = []
    for sid in sorted(meta_df['survey_id'].unique()):
        sdata = meta_df[meta_df['survey_id'] == sid]
        w = sdata['weight'].values
        c = sdata['pred'].values
        tw = w.sum()

        row = {'survey_id': int(sid)}
        for t in thresholds:
            row[f"pct_hh_below_{t:.2f}"] = np.sum(w * (c < t)) / tw
        results.append(row)

    return pd.DataFrame(results)

def create_submission(predictions, test_meta, poverty_rates, output_dir, thresholds):
    """Create submission files."""
    print("\n" + "="*60)
    print("STEP 6: CREATING SUBMISSION")
    print("="*60)

    # Consumption file
    cons_df = pd.DataFrame({
        'survey_id': test_meta['survey_id'].astype(int),
        'hhid': test_meta['hhid'].astype(int),
        'cons_ppp17': predictions
    }).sort_values(['survey_id', 'hhid'])

    cons_path = output_dir / "predicted_household_consumption.csv"
    cons_df.to_csv(cons_path, index=False)
    print(f"Saved: {cons_path}")

    # Poverty rates
    cols = ['survey_id'] + [f"pct_hh_below_{t:.2f}" for t in thresholds]
    pov_df = poverty_rates[cols].sort_values('survey_id')
    pov_path = output_dir / "predicted_poverty_distribution.csv"
    pov_df.to_csv(pov_path, index=False)
    print(f"Saved: {pov_path}")

    # ZIP
    zip_path = output_dir / "submission.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(cons_path, "predicted_household_consumption.csv")
        z.write(pov_path, "predicted_poverty_distribution.csv")

    print(f"Saved: {zip_path}")
    return zip_path

# ================================================================================
# MAIN
# ================================================================================
def main():
    print("="*60)
    print("SOLUTION V3 - NEURAL NETWORK + LIGHTGBM ENSEMBLE")
    print("="*60)

    # Load data
    train_data, test_data, train_rates = load_data()

    # Feature engineering
    train_proc, test_proc = engineer_features(train_data.copy(), test_data.copy())

    # Prepare data
    (X_train, X_train_scaled, y_train, y_train_log,
     X_test, X_test_scaled, train_meta, test_meta, features) = prepare_data(train_proc, test_proc)

    # Train ensemble
    nn_models, lgb_models = train_ensemble(
        X_train, X_train_scaled, y_train_log, y_train, features, Config
    )

    # Predict
    predictions = predict_ensemble(nn_models, lgb_models, X_test, X_test_scaled, Config)

    # Poverty rates
    poverty_rates = calc_poverty_rates(predictions, test_meta, Config.THRESHOLDS)

    print("\nTest poverty rates at $7.70:")
    for _, row in poverty_rates.iterrows():
        print(f"  Survey {int(row['survey_id'])}: {row['pct_hh_below_7.70']:.4f}")

    # Create submission
    zip_path = create_submission(predictions, test_meta, poverty_rates, Config.OUTPUT_DIR, Config.THRESHOLDS)

    print("\n" + "="*60)
    print("V3 ENSEMBLE SOLUTION COMPLETE!")
    print("="*60)
    print(f"\nSubmission ready: {zip_path}")

    return nn_models, lgb_models, predictions

if __name__ == "__main__":
    main()


In [ ]:
"""
================================================================================
SOLUTION V4 - THE TRINITY (NN + LIGHTGBM + XGBOOST + PCA)
================================================================================
"""

import os
import warnings
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ================================================================================
# CONFIGURATION
# ================================================================================
class Config:
    # ADJUST YOUR PATHS HERE IF NEEDED
    DATA_DIR = Path("/kaggle/input/twb-poverty-prediction/TWB") 
    OUTPUT_DIR = Path("/kaggle/working/")

    THRESHOLDS = [3.17, 3.94, 4.60, 5.26, 5.88, 6.47, 7.06, 7.70, 8.40, 
                  9.13, 9.87, 10.70, 11.62, 12.69, 14.03, 15.64, 17.76, 20.99, 27.37]

    # Neural Network params
    NN_BATCH_SIZE = 512
    NN_LR = 0.001
    NN_EPOCHS = 40
    NN_EARLY_STOP = 10

    # LightGBM params
    LGB_PARAMS = {
        'objective': 'regression', 'metric': 'mae', 'boosting_type': 'gbdt',
        'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
        'verbose': -1, 'seed': SEED, 'n_jobs': -1
    }

    # XGBoost params (NEW)
    XGB_PARAMS = {
        'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
        'max_depth': 8, 'learning_rate': 0.02, 'subsample': 0.7,
        'colsample_bytree': 0.7, 'n_jobs': -1, 'random_state': SEED,
        'tree_method': 'hist' # Fast training
    }

    N_FOLDS = 5

    # THE TRINITY WEIGHTS
    NN_WEIGHT = 0.30   # 30%
    LGB_WEIGHT = 0.40  # 40%
    XGB_WEIGHT = 0.30  # 30%

    USE_LOG_TARGET = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {Config.DEVICE}")
print(f"Strategy: {Config.NN_WEIGHT} NN + {Config.LGB_WEIGHT} LGB + {Config.XGB_WEIGHT} XGB")

# ================================================================================
# DATA LOADING
# ================================================================================
def load_data():
    print("\n" + "="*60 + "\nSTEP 1: LOADING DATA\n" + "="*60)
    # Handle different file names if necessary
    try:
        train_feat = pd.read_csv(Config.DATA_DIR / "train_hh_features.csv")
        test_feat = pd.read_csv(Config.DATA_DIR / "test_hh_features.csv")
        train_gt = pd.read_csv(Config.DATA_DIR / "train_hh_gt.csv")
    except FileNotFoundError:
        print("Error: Files not found in 'data/' folder. Please check path.")
        return None, None, None

    # Rename hhid -> household_id if needed
    if 'hhid' in train_feat.columns: train_feat.rename(columns={'hhid': 'household_id'}, inplace=True)
    if 'hhid' in test_feat.columns: test_feat.rename(columns={'hhid': 'household_id'}, inplace=True)
    if 'hhid' in train_gt.columns: train_gt.rename(columns={'hhid': 'household_id'}, inplace=True)

    train_data = train_feat.merge(train_gt, on=['household_id'], suffixes=('', '_gt'))
    
    # Clean up duplicate columns from merge
    if 'survey_id_gt' in train_data.columns:
        train_data['survey_id'] = train_data['survey_id_gt']
        train_data.drop(columns=['survey_id_gt'], inplace=True)

    print(f"Train Shape: {train_data.shape}, Test Shape: {test_feat.shape}")
    return train_data, test_feat

# ================================================================================
# FEATURE ENGINEERING (UPGRADED)
# ================================================================================
def convert_val(val):
    if pd.isna(val): return 0
    s = str(val).strip().lower()
    return 1 if s in ['yes', '1', '1.0', 'true', 'access'] else 0

def engineer_features(train_data, test_data):
    print("\n" + "="*60 + "\nSTEP 2: FEATURE ENGINEERING (PCA + CLUSTERS)\n" + "="*60)
    
    train_data['is_train'] = 1
    test_data['is_train'] = 0
    
    # Add missing columns to test
    for col in train_data.columns:
        if col not in test_data.columns and col not in ['cons_ppp17', 'is_train']:
            test_data[col] = np.nan
            
    combined = pd.concat([train_data, test_data], ignore_index=True)

    # 1. CLEAN BINARY COLUMNS
    consumed_cols = [c for c in combined.columns if 'consumed' in c]
    binary_cols = ['water', 'toilet', 'sewer', 'elect', 'owner', 'male', 'employed', 'any_nonagric', 'urban']
    
    # Only process columns that actually exist
    existing_bin_cols = [c for c in binary_cols if c in combined.columns]
    
    for col in consumed_cols + existing_bin_cols:
        combined[col] = combined[col].apply(convert_val)

    # 2. NUMERIC CLEANING
    num_cols = ['hsize', 'utl_exp_ppp17', 'age', 'weight'] + \
               [c for c in combined.columns if 'num_' in c or 'share_' in c]
               
    for col in num_cols:
        if col in combined.columns:
            combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(0)

    # === NEW: PCA WEALTH INDEX ===
    # We take all asset/consumption booleans to find the "Hidden Wealth Factor"
    pca_cols = consumed_cols + existing_bin_cols
    pca_data = combined[pca_cols].fillna(0)
    
    print("   Running PCA on assets...")
    pca = PCA(n_components=1)
    combined['wealth_index'] = pca.fit_transform(pca_data)
    
    # === NEW: RELATIVE WEALTH ===
    # "Are you rich compared to your neighbors?"
    if 'region1' in combined.columns:
        combined['region_wealth_mean'] = combined.groupby('region1')['wealth_index'].transform('mean')
        combined['relative_wealth'] = combined['wealth_index'] - combined['region_wealth_mean']

    # 3. EXISTING FEATURES (Kept from V3)
    combined['total_food'] = combined[consumed_cols].sum(axis=1)
    combined['infra_score'] = combined[[c for c in ['water', 'elect', 'toilet'] if c in combined.columns]].sum(axis=1)
    
    if 'utl_exp_ppp17' in combined.columns and 'hsize' in combined.columns:
        combined['log_utl'] = np.log1p(combined['utl_exp_ppp17'])
        combined['utl_per_capita'] = combined['utl_exp_ppp17'] / (combined['hsize'] + 1)

    # 4. ENCODE CATEGORICALS
    cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
    for col in cat_cols:
        if col != 'is_train':
            combined[col] = combined[col].fillna('Unknown')
            combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

    # Split back
    train_proc = combined[combined['is_train'] == 1].drop(columns=['is_train'])
    test_proc = combined[combined['is_train'] == 0].drop(columns=['is_train', 'cons_ppp17'], errors='ignore')
    
    print(f"   Features created: {train_proc.shape[1]}")
    return train_proc, test_proc

# ================================================================================
# PREPARE FOR MODELING
# ================================================================================
def prepare_data(train_data, test_data):
    exclude = ['household_id', 'hhid', 'survey_id', 'cons_ppp17', 'com', 'sample']
    features = [c for c in train_data.columns if c not in exclude]
    
    X_train = train_data[features].values.astype(np.float32)
    y_train = train_data['cons_ppp17'].values.astype(np.float32)
    X_test = test_data[features].values.astype(np.float32)
    
    # Log Transform Target
    y_train_log = np.log1p(y_train) if Config.USE_LOG_TARGET else y_train
    
    # Scale for NN
    scaler = StandardScaler()
    X_train_scaled = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_scaled = np.nan_to_num(scaler.transform(X_test), 0)
    
    # Metadata for Submission
    # Use 'hhid' or 'household_id' whichever exists for ID
    id_col = 'household_id' if 'household_id' in test_data.columns else 'hhid'
    
    test_meta = test_data[['survey_id', id_col, 'weight']].copy()
    if id_col != 'household_id': test_meta.rename(columns={id_col: 'household_id'}, inplace=True)
    
    return X_train, X_train_scaled, y_train, y_train_log, X_test, X_test_scaled, test_meta, features

# ================================================================================
# MODELS
# ================================================================================
class PovertyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_nn_fold(X_tr, y_tr, X_val, y_val, input_dim, config):
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)), 
                              batch_size=config.NN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)), 
                            batch_size=config.NN_BATCH_SIZE)
    
    model = PovertyNet(input_dim).to(config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=config.NN_LR)
    criterion = nn.L1Loss() # MAE Loss
    
    best_loss = float('inf')
    patience = 0
    best_state = None
    
    for epoch in range(config.NN_EPOCHS):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(config.DEVICE), by.to(config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(config.DEVICE), by.to(config.DEVICE)
                val_loss += criterion(model(bx), by).item() * len(bx)
        val_loss /= len(y_val)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict().copy()
            patience = 0
        else:
            patience += 1
            if patience >= config.NN_EARLY_STOP: break
            
    model.load_state_dict(best_state)
    return model

# ================================================================================
# MAIN TRAINING LOOP (TRINITY)
# ================================================================================
def train_and_predict():
    train_data, test_data = load_data()
    if train_data is None: return

    train_proc, test_proc = engineer_features(train_data, test_data)
    
    X_train, X_train_scaled, y_train, y_train_log, \
    X_test, X_test_scaled, test_meta, features = prepare_data(train_proc, test_proc)
    
    kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=SEED)
    
    # Store predictions for the test set
    final_preds = np.zeros(len(X_test))
    
    print("\n" + "="*60 + "\nSTEP 3: TRAINING THE TRINITY\n" + "="*60)
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\n--- Fold {fold+1}/{Config.N_FOLDS} ---")
        
        # Split
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        X_tr_s, X_val_s = X_train_scaled[tr_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_log[tr_idx], y_train_log[val_idx]
        y_val_orig = y_train[val_idx]
        
        # 1. Neural Network
        print("   Training NN...", end=" ")
        nn_model = train_nn_fold(X_tr_s, y_tr, X_val_s, y_val, X_tr_s.shape[1], Config)
        with torch.no_grad():
            nn_val = nn_model(torch.FloatTensor(X_val_s).to(Config.DEVICE)).cpu().numpy()
            nn_test = nn_model(torch.FloatTensor(X_test_scaled).to(Config.DEVICE)).cpu().numpy()
        print(f"Done.")

        # 2. LightGBM
        print("   Training LightGBM...", end=" ")
        lgb_train = lgb.Dataset(X_tr, y_tr, feature_name=features)
        lgb_val = lgb.Dataset(X_val, y_val, feature_name=features, reference=lgb_train)
        lgb_model = lgb.train(Config.LGB_PARAMS, lgb_train, num_boost_round=1000, 
                              valid_sets=[lgb_val], callbacks=[lgb.early_stopping(50, verbose=False)])
        lgb_val_pred = lgb_model.predict(X_val)
        lgb_test_pred = lgb_model.predict(X_test)
        print(f"Done.")

        # 3. XGBoost
        print("   Training XGBoost...", end=" ")
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        dtest = xgb.DMatrix(X_test)
        xgb_model = xgb.train(Config.XGB_PARAMS, dtrain, num_boost_round=1000, 
                              evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        xgb_val_pred = xgb_model.predict(dval)
        xgb_test_pred = xgb_model.predict(dtest)
        print(f"Done.")
        
        # Ensemble Validation Score
        ens_val = (Config.NN_WEIGHT * nn_val + Config.LGB_WEIGHT * lgb_val_pred + Config.XGB_WEIGHT * xgb_val_pred)
        if Config.USE_LOG_TARGET: ens_val = np.expm1(ens_val)
        
        mae = mean_absolute_error(y_val_orig, np.maximum(ens_val, 0))
        print(f"   >>> FOLD MAE: {mae:.4f}")
        
        # Accumulate Test Predictions
        fold_test_pred = (Config.NN_WEIGHT * nn_test + Config.LGB_WEIGHT * lgb_test_pred + Config.XGB_WEIGHT * xgb_test_pred)
        if Config.USE_LOG_TARGET: fold_test_pred = np.expm1(fold_test_pred)
        
        final_preds += fold_test_pred / Config.N_FOLDS

    return final_preds, test_meta

# ================================================================================
# SUBMISSION GENERATION
# ================================================================================
def generate_submission(predictions, meta_df):
    print("\n" + "="*60 + "\nSTEP 4: GENERATING SUBMISSION\n" + "="*60)
    
    # 1. Consumption File
    meta_df['cons_ppp17'] = np.maximum(predictions, 0) # No negative money
    
    sub_cons = meta_df[['survey_id', 'household_id', 'cons_ppp17']].copy()
    sub_cons.columns = ['survey_id', 'household_id', 'per_capita_household_consumption']
    sub_cons.to_csv(Config.OUTPUT_DIR / 'predicted_household_consumption.csv', index=False)
    
    # 2. Poverty Rates (Weighted)
    results = []
    for sid in meta_df['survey_id'].unique():
        sdata = meta_df[meta_df['survey_id'] == sid]
        w = sdata['weight'].values
        c = sdata['cons_ppp17'].values
        total_w = w.sum()
        
        row = {'survey_id': int(sid)}
        for t in Config.THRESHOLDS:
            # Sum weights of people below threshold
            poor_w = np.sum(w[c < t])
            row[f"pct_hh_below_{t}"] = poor_w / total_w
        results.append(row)
        
    sub_pov = pd.DataFrame(results)
    # Reorder columns to match submission format strictly
    cols = ['survey_id'] + [f"pct_hh_below_{t}" for t in Config.THRESHOLDS]
    sub_pov = sub_pov[cols]
    sub_pov.to_csv(Config.OUTPUT_DIR / 'predicted_poverty_distribution.csv', index=False)
    
    # 3. Zip
    zip_path = Config.OUTPUT_DIR / "submission_v4.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(Config.OUTPUT_DIR / 'predicted_household_consumption.csv', 'predicted_household_consumption.csv')
        z.write(Config.OUTPUT_DIR / 'predicted_poverty_distribution.csv', 'predicted_poverty_distribution.csv')
        
    print(f"✅ SUCCESS! Created: {zip_path}")

if __name__ == "__main__":
    preds, meta = train_and_predict()
    if preds is not None:
        generate_submission(preds, meta)

In [ ]:
def apply_calibration_and_submit(final_preds, train_data, test_meta, output_dir, thresholds):
    print("\n" + "="*60)
    print("STEP 5: CALIBRATION (THE FINAL TRICK)")
    print("="*60)
    
    # 1. Get Statistics
    # We look at the REAL training consumption
    y_train = train_data['cons_ppp17'].values
    train_mean = np.mean(y_train)
    train_std = np.std(y_train)
    
    # We look at your MODEL'S predicted consumption
    pred_mean = np.mean(final_preds)
    pred_std = np.std(final_preds)
    
    print(f"Original Train Data: Mean={train_mean:.4f}, Std={train_std:.4f}")
    print(f"Your Predictions:    Mean={pred_mean:.4f}, Std={pred_std:.4f}")
    
    # 2. The Calibration Formula
    # We shift the mean and stretch the standard deviation to match reality
    calibrated_preds = (final_preds - pred_mean) * (train_std / pred_std) + train_mean
    
    # Enforce constraints (money cannot be negative)
    calibrated_preds = np.maximum(calibrated_preds, 0)
    
    print(f"Calibrated Preds:    Mean={np.mean(calibrated_preds):.4f}, Std={np.std(calibrated_preds):.4f}")
    print("(Notice how the Std now matches the Train Data!)")
    
    # 3. Generate Submission with CALIBRATED values
    print("\nGenerating Final Submission...")
    
    # Consumption File
    test_meta['cons_ppp17'] = calibrated_preds
    sub_cons = test_meta[['survey_id', 'household_id', 'cons_ppp17']].copy()
    sub_cons.columns = ['survey_id', 'household_id', 'per_capita_household_consumption']
    sub_cons.to_csv(output_dir / 'predicted_household_consumption.csv', index=False)
    
    # Poverty Rates File
    results = []
    for sid in test_meta['survey_id'].unique():
        sdata = test_meta[test_meta['survey_id'] == sid]
        w = sdata['weight'].values
        c = sdata['cons_ppp17'].values
        total_w = w.sum()
        
        row = {'survey_id': int(sid)}
        for t in thresholds:
            # The calibration helps here! It pushes more people below the threshold.
            poor_w = np.sum(w[c < t])
            row[f"pct_hh_below_{t}"] = poor_w / total_w
        results.append(row)
        
    sub_pov = pd.DataFrame(results)
    cols = ['survey_id'] + [f"pct_hh_below_{t}" for t in thresholds]
    sub_pov = sub_pov[cols]
    sub_pov.to_csv(output_dir / 'predicted_poverty_distribution.csv', index=False)
    
    # Zip
    zip_path = output_dir / "submission_final_calibrated.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(output_dir / 'predicted_household_consumption.csv', 'predicted_household_consumption.csv')
        z.write(output_dir / 'predicted_poverty_distribution.csv', 'predicted_poverty_distribution.csv')
        
    print(f"✅ FINAL SUBMISSION READY: {zip_path}")
    return zip_path

# --- HOW TO RUN THIS ---
# Copy the code above.
# call it at the very end of your script:
apply_calibration_and_submit(preds, train_data, meta, Config.OUTPUT_DIR, Config.THRESHOLDS)

In [ ]:
"""
================================================================================
SOLUTION V5 - THE WEIGHTED TRINITY (NN + LGBM + XGB) + CALIBRATION
================================================================================
"""

import os
import warnings
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ================================================================================
# CONFIGURATION
# ================================================================================
class Config:
    DATA_DIR = Path("/kaggle/input/twb-poverty-prediction/TWB") 
    OUTPUT_DIR = Path("/kaggle/working/")

    THRESHOLDS = [3.17, 3.94, 4.60, 5.26, 5.88, 6.47, 7.06, 7.70, 8.40, 
                  9.13, 9.87, 10.70, 11.62, 12.69, 14.03, 15.64, 17.76, 20.99, 27.37]

    # Neural Network
    NN_BATCH_SIZE = 512
    NN_LR = 0.001
    NN_EPOCHS = 40
    NN_EARLY_STOP = 10

    # LightGBM
    LGB_PARAMS = {
        'objective': 'regression', 'metric': 'mae', 'boosting_type': 'gbdt',
        'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
        'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
        'verbose': -1, 'seed': SEED, 'n_jobs': -1
    }

    # XGBoost
    XGB_PARAMS = {
        'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
        'max_depth': 8, 'learning_rate': 0.02, 'subsample': 0.7,
        'colsample_bytree': 0.7, 'n_jobs': -1, 'random_state': SEED,
        'tree_method': 'hist'
    }

    N_FOLDS = 5

    # ENSEMBLE WEIGHTS
    NN_WEIGHT = 0.30
    LGB_WEIGHT = 0.40
    XGB_WEIGHT = 0.30

    USE_LOG_TARGET = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"🔥 Running V5 Final Solution on {Config.DEVICE}...")

# ================================================================================
# DATA LOADING
# ================================================================================
def load_and_fix(path):
    df = pd.read_csv(path)
    if 'hhid' in df.columns:
        df = df.rename(columns={'hhid': 'household_id'})
    return df

def load_data():
    print("\n--- STEP 1: LOADING DATA ---")
    try:
        train_feat = load_and_fix(Config.DATA_DIR / "train_hh_features.csv")
        test_feat = load_and_fix(Config.DATA_DIR / "test_hh_features.csv")
        train_gt = load_and_fix(Config.DATA_DIR / "train_hh_gt.csv")
    except FileNotFoundError:
        print("❌ Error: Files not found. Check 'data/' folder.")
        return None, None

    train_data = train_feat.merge(train_gt, on=['household_id'], suffixes=('', '_gt'))
    if 'survey_id_gt' in train_data.columns:
        train_data['survey_id'] = train_data['survey_id_gt']
        train_data.drop(columns=['survey_id_gt'], inplace=True)

    print(f"Train Shape: {train_data.shape}, Test Shape: {test_feat.shape}")
    return train_data, test_feat

# ================================================================================
# FEATURE ENGINEERING
# ================================================================================
def convert_val(val):
    if pd.isna(val): return 0
    s = str(val).strip().lower()
    return 1 if s in ['yes', '1', '1.0', 'true', 'access'] else 0

def engineer_features(train_data, test_data):
    print("\n--- STEP 2: FEATURE ENGINEERING (PCA + WEALTH) ---")
    
    train_data['is_train'] = 1
    test_data['is_train'] = 0
    
    # Align columns
    for col in train_data.columns:
        if col not in test_data.columns and col not in ['cons_ppp17', 'is_train']:
            test_data[col] = np.nan
            
    combined = pd.concat([train_data, test_data], ignore_index=True)

    # 1. Clean Binary Columns
    consumed_cols = [c for c in combined.columns if 'consumed' in c]
    binary_cols = ['water', 'toilet', 'sewer', 'elect', 'owner', 'male', 'employed', 'any_nonagric', 'urban']
    existing_bin_cols = [c for c in binary_cols if c in combined.columns]
    
    for col in consumed_cols + existing_bin_cols:
        combined[col] = combined[col].apply(convert_val)

    # 2. Clean Numeric Columns
    num_cols = ['hsize', 'utl_exp_ppp17', 'age', 'weight'] + \
               [c for c in combined.columns if 'num_' in c or 'share_' in c]
    for col in num_cols:
        if col in combined.columns:
            combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(0)

    # 3. PCA Wealth Index (The "Richness" Factor)
    pca_cols = consumed_cols + existing_bin_cols
    pca_data = combined[pca_cols].fillna(0)
    
    pca = PCA(n_components=1)
    combined['wealth_index'] = pca.fit_transform(pca_data)
    
    # 4. Relative Wealth (Rich for your region?)
    if 'region1' in combined.columns:
        combined['region_wealth_mean'] = combined.groupby('region1')['wealth_index'].transform('mean')
        combined['relative_wealth'] = combined['wealth_index'] - combined['region_wealth_mean']

    # 5. Standard Aggregates
    combined['total_food'] = combined[consumed_cols].sum(axis=1)
    combined['infra_score'] = combined[[c for c in ['water', 'elect', 'toilet'] if c in combined.columns]].sum(axis=1)
    
    if 'utl_exp_ppp17' in combined.columns and 'hsize' in combined.columns:
        combined['log_utl'] = np.log1p(combined['utl_exp_ppp17'])
        combined['utl_per_capita'] = combined['utl_exp_ppp17'] / (combined['hsize'] + 1)

    # 6. Categorical Encoding
    cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
    for col in cat_cols:
        if col != 'is_train':
            combined[col] = combined[col].fillna('Unknown')
            combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

    train_proc = combined[combined['is_train'] == 1].drop(columns=['is_train'])
    test_proc = combined[combined['is_train'] == 0].drop(columns=['is_train', 'cons_ppp17'], errors='ignore')
    
    return train_proc, test_proc

# ================================================================================
# PREPARE FOR MODELING (WITH WEIGHTS)
# ================================================================================
def prepare_data(train_data, test_data):
    # CRITICAL: Do not include weight in features, but keep it for training
    exclude = ['household_id', 'survey_id', 'cons_ppp17', 'com', 'sample', 'weight']
    features = [c for c in train_data.columns if c not in exclude]
    
    X_train = train_data[features].values.astype(np.float32)
    y_train = train_data['cons_ppp17'].values.astype(np.float32)
    w_train = train_data['weight'].values.astype(np.float32) # Weights for Loss Function
    
    X_test = test_data[features].values.astype(np.float32)
    
    y_train_log = np.log1p(y_train) if Config.USE_LOG_TARGET else y_train
    
    scaler = StandardScaler()
    X_train_scaled = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_scaled = np.nan_to_num(scaler.transform(X_test), 0)
    
    test_meta = test_data[['survey_id', 'household_id', 'weight']].copy()
    
    return X_train, X_train_scaled, y_train, y_train_log, w_train, X_test, X_test_scaled, test_meta, features

# ================================================================================
# NEURAL NETWORK
# ================================================================================
class PovertyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_nn_weighted(X_tr, y_tr, w_tr, X_val, y_val, w_val, input_dim, config):
    # Create datasets with weights
    train_ds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr), torch.FloatTensor(w_tr))
    val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val), torch.FloatTensor(w_val))
    
    train_loader = DataLoader(train_ds, batch_size=config.NN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.NN_BATCH_SIZE)
    
    model = PovertyNet(input_dim).to(config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=config.NN_LR)
    criterion = nn.L1Loss(reduction='none') # Manual weighting
    
    best_loss = float('inf')
    patience = 0
    best_state = None
    
    for epoch in range(config.NN_EPOCHS):
        model.train()
        for bx, by, bw in train_loader:
            bx, by, bw = bx.to(config.DEVICE), by.to(config.DEVICE), bw.to(config.DEVICE)
            optimizer.zero_grad()
            loss = (criterion(model(bx), by) * bw).mean() # Weighted Loss
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0
        total_w = 0
        with torch.no_grad():
            for bx, by, bw in val_loader:
                bx, by, bw = bx.to(config.DEVICE), by.to(config.DEVICE), bw.to(config.DEVICE)
                val_loss += (criterion(model(bx), by) * bw).sum().item()
                total_w += bw.sum().item()
        val_loss /= total_w
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict().copy()
            patience = 0
        else:
            patience += 1
            if patience >= config.NN_EARLY_STOP: break
            
    model.load_state_dict(best_state)
    return model

# ================================================================================
# MAIN TRAINING LOOP
# ================================================================================
def train_ensemble():
    train_data, test_data = load_data()
    if train_data is None: return None, None, None

    train_proc, test_proc = engineer_features(train_data, test_data)
    
    X_train, X_train_scaled, y_train, y_train_log, w_train, \
    X_test, X_test_scaled, test_meta, features = prepare_data(train_proc, test_proc)
    
    kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=SEED)
    final_preds = np.zeros(len(X_test))
    
    print("\n--- STEP 3: TRAINING ENSEMBLE (WEIGHTED) ---")
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\n>> FOLD {fold+1}/{Config.N_FOLDS}")
        
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        X_tr_s, X_val_s = X_train_scaled[tr_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_log[tr_idx], y_train_log[val_idx]
        w_tr, w_val = w_train[tr_idx], w_train[val_idx]
        y_val_orig = y_train[val_idx]
        
        # 1. NN
        print("   [1/3] Neural Net...", end=" ")
        nn_model = train_nn_weighted(X_tr_s, y_tr, w_tr, X_val_s, y_val, w_val, X_tr_s.shape[1], Config)
        with torch.no_grad():
            nn_test = nn_model(torch.FloatTensor(X_test_scaled).to(Config.DEVICE)).cpu().numpy()
            nn_val_pred = nn_model(torch.FloatTensor(X_val_s).to(Config.DEVICE)).cpu().numpy()
        print("Done.")
        
        # 2. LightGBM (Weighted)
        print("   [2/3] LightGBM...", end=" ")
        lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=features)
        lgb_val = lgb.Dataset(X_val, y_val, weight=w_val, feature_name=features, reference=lgb_train)
        lgb_model = lgb.train(Config.LGB_PARAMS, lgb_train, num_boost_round=1000, 
                              valid_sets=[lgb_val], callbacks=[lgb.early_stopping(50, verbose=False)])
        lgb_test = lgb_model.predict(X_test)
        lgb_val_pred = lgb_model.predict(X_val)
        print("Done.")
        
        # 3. XGBoost (Weighted)
        print("   [3/3] XGBoost...", end=" ")
        dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
        dval = xgb.DMatrix(X_val, label=y_val, weight=w_val)
        dtest = xgb.DMatrix(X_test)
        xgb_model = xgb.train(Config.XGB_PARAMS, dtrain, num_boost_round=1000, 
                              evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
        xgb_test = xgb_model.predict(dtest)
        xgb_val_pred = xgb_model.predict(dval)
        print("Done.")
        
        # Check Validation Score (Weighted MAE)
        ens_val = (Config.NN_WEIGHT * nn_val_pred + Config.LGB_WEIGHT * lgb_val_pred + Config.XGB_WEIGHT * xgb_val_pred)
        if Config.USE_LOG_TARGET: ens_val = np.expm1(ens_val)
        
        # Manual Weighted MAE Calculation
        w_mae = np.sum(w_val * np.abs(ens_val - y_val_orig)) / np.sum(w_val)
        print(f"   >>> Weighted MAE: {w_mae:.4f}")
        
        # Average Test Predictions
        fold_test = (Config.NN_WEIGHT * nn_test + Config.LGB_WEIGHT * lgb_test + Config.XGB_WEIGHT * xgb_test)
        if Config.USE_LOG_TARGET: fold_test = np.expm1(fold_test)
        final_preds += fold_test / Config.N_FOLDS

    return final_preds, test_meta, y_train

# ================================================================================
# CALIBRATION & SUBMISSION
# ================================================================================
def apply_calibration(preds, y_train_orig):
    print("\n--- STEP 4: CALIBRATION ---")
    train_mean, train_std = np.mean(y_train_orig), np.std(y_train_orig)
    pred_mean, pred_std = np.mean(preds), np.std(preds)
    
    print(f"Train Stats: Mean={train_mean:.2f}, Std={train_std:.2f}")
    print(f"Pred Stats:  Mean={pred_mean:.2f}, Std={pred_std:.2f}")
    
    # Scale spread to match training data (catches extreme poor)
    calibrated = (preds - pred_mean) * (train_std / pred_std) + train_mean
    calibrated = np.maximum(calibrated, 0) # No negative spending
    
    print(f"Calibrated:  Mean={np.mean(calibrated):.2f}, Std={np.std(calibrated):.2f}")
    return calibrated

def generate_submission(predictions, meta_df):
    print("\n--- STEP 5: SAVING ---")
    meta_df['cons_ppp17'] = predictions
    
    # File 1: Consumption
    sub_cons = meta_df[['survey_id', 'household_id', 'cons_ppp17']].copy()
    sub_cons.columns = ['survey_id', 'household_id', 'per_capita_household_consumption']
    sub_cons.to_csv(Config.OUTPUT_DIR / 'predicted_household_consumption.csv', index=False)
    
    # File 2: Poverty Rates
    results = []
    for sid in meta_df['survey_id'].unique():
        sdata = meta_df[meta_df['survey_id'] == sid]
        w = sdata['weight'].values
        c = sdata['cons_ppp17'].values
        total_w = w.sum()
        
        row = {'survey_id': int(sid)}
        for t in Config.THRESHOLDS:
            poor_w = np.sum(w[c < t])
            row[f"pct_hh_below_{t}"] = poor_w / total_w
        results.append(row)
        
    sub_pov = pd.DataFrame(results)
    cols = ['survey_id'] + [f"pct_hh_below_{t}" for t in Config.THRESHOLDS]
    sub_pov = sub_pov[cols]
    sub_pov.to_csv(Config.OUTPUT_DIR / 'predicted_poverty_distribution.csv', index=False)
    
    # Zip
    zip_path = Config.OUTPUT_DIR / "submission_v5_final.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(Config.OUTPUT_DIR / 'predicted_household_consumption.csv', 'predicted_household_consumption.csv')
        z.write(Config.OUTPUT_DIR / 'predicted_poverty_distribution.csv', 'predicted_poverty_distribution.csv')
        
    print(f"✅ DONE: {zip_path}")

if __name__ == "__main__":
    preds, meta, y_train = train_ensemble()
    if preds is not None:
        # Final Polish: Calibrate predictions to match reality
        calibrated_preds = apply_calibration(preds, y_train)
        generate_submission(calibrated_preds, meta)